# Episode 2 — CNN on MNIST

**Companion notebook for [Neural Architecture Tour — Episode 2](https://provandal.github.io/neural-architecture-tour/02-cnn/)**

---

### What This Notebook Does

This notebook trains a small convolutional neural network to recognize handwritten digits (MNIST). Along the way it exports two artifacts that the browser app consumes:

1. **`precomputed.json`** — training history, filter snapshots across training, and sample feature maps.
2. **`model.onnx`** — the trained model, exported to ONNX so it can run directly in the browser via `@huggingface/transformers`.

### What You'll Learn

- How a convolution *actually* works — the math, not the metaphor.
- How filters evolve from random noise to edge detectors during training.
- How to train a model in PyTorch, inspect its weights, and export it for production use.
- How ONNX lets you take a model out of Python and run it somewhere else entirely (like a browser).

### Prerequisites

- **Google Account** (to run this in Colab). Runs locally too if you have PyTorch installed.
- **GPU is optional** — this model trains in about 60 seconds on CPU, 15 seconds on a free T4.
- **No ML experience required** — every step is explained.
- **Runtime:** ~3–5 minutes total.

### How to Use This Notebook

Click into each gray code cell and press **Shift+Enter** to run it. Run cells in order from top to bottom. If a cell fails, re-run the cell above it first — sometimes Colab's state resets between sessions.

---

## Step 1: Install Dependencies

Colab already has PyTorch and matplotlib preinstalled, but we need a couple of extras for ONNX export and validation:

- **`onnx`** — the open standard for portable ML model formats. Saving a model as ONNX means any runtime (Python, JavaScript, C++, iOS, Android) can load and run it without needing PyTorch itself.
- **`onnxruntime`** — Microsoft's inference engine for ONNX models. We use it below to *verify* that the exported ONNX file produces the same predictions as the original PyTorch model.

The `-q` flag is just "quiet" — it suppresses the noisy pip output.

In [ ]:
!pip install -q onnx onnxruntime

## Step 2: Imports and Environment Check

Standard PyTorch imports, plus a couple of helpers. We also check whether a GPU is available. If you're on Colab, go to **Runtime → Change runtime type → T4 GPU** to enable one — training is ~4x faster. It's not required, but it's free and makes iteration pleasant.

In [ ]:
import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 3: Load MNIST

MNIST is the "hello world" of computer vision: 70,000 grayscale images of handwritten digits 0–9, each 28×28 pixels. 60,000 images are for training, 10,000 for testing.

The `transforms.ToTensor()` step converts each 28×28 image into a PyTorch tensor with values in [0, 1]. The `Normalize` step shifts those values so they have the standard mean (0.1307) and standard deviation (0.3081) that MNIST is usually trained with — this makes training more stable.

After loading, we plot 10 sample digits so you can see what the model is looking at.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False)

print(f'Training images: {len(train_data):,}')
print(f'Test images:     {len(test_data):,}')
print(f'Image shape:     {train_data[0][0].shape}')

# Show a few samples
fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
    img, label = train_data[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(str(label))
    ax.axis('off')
plt.suptitle('Sample MNIST digits', y=1.05)
plt.tight_layout()
plt.show()

## Step 4: Define the CNN

Our architecture is deliberately small — the goal is to be able to visualize every layer, not to hit state-of-the-art accuracy.

```
Input: 1 × 28 × 28                    (one grayscale image)
  ↓ Conv2d(1 → 8, kernel 3×3, pad 1)  (8 filters, each learns one pattern)
  ↓ ReLU
  ↓ MaxPool2d(2)                      → 8 × 14 × 14
  ↓ Conv2d(8 → 16, kernel 3×3, pad 1) (16 filters, looking at 8 previous channels)
  ↓ ReLU
  ↓ MaxPool2d(2)                      → 16 × 7 × 7
  ↓ Flatten                           → 784
  ↓ Linear(784 → 64)
  ↓ ReLU
  ↓ Linear(64 → 10)                   → 10 class logits
```

**Why these choices?**

- **8 filters in the first conv layer.** Small enough to visualize in a single row. Large enough to learn useful edge detectors.
- **3×3 kernel.** The smallest useful kernel; most modern CNNs (ResNet, VGG) use 3×3 almost exclusively.
- **padding=1.** Keeps the spatial dimensions the same after convolution (28×28 in, 28×28 out). Without padding, edges get chopped off.
- **MaxPool.** Halves the spatial dimensions after each block. This is how the model gets a wider "receptive field" over the image — deeper layers see larger patches.

Total trainable parameters: about 55,000. Compare that to a Transformer (millions to billions). Small, fast, effective.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(16 * 7 * 7, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # [B, 8, 14, 14]
        x = self.pool(F.relu(self.conv2(x)))   # [B, 16, 7, 7]
        x = x.flatten(1)                       # [B, 784]
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = SmallCNN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTotal trainable parameters: {n_params:,}')

## Step 5: Train the Model

Training is one loop: for each batch of images, compute predictions, compute loss, compute gradients, take a tiny step in the direction that reduces loss. Repeat. We train for 5 epochs, which is plenty for MNIST — accuracy saturates quickly.

**The interesting part for Episode 2:** we snapshot the first-layer filters at specific training steps. The browser app plays these back as an animation, so you can literally watch random noise turn into edge detectors. We save snapshots at step 0 (pure random init), then at a handful of steps during training, then at the end.

Optimizer: **Adam** with a learning rate of 1e-3. Adam adapts per-parameter learning rates — it's the default choice for most modern training and it Just Works for small models.

Loss: **Cross-entropy**. The standard for classification. Low loss = model is confident and correct. High loss = model is uncertain or wrong.

You should see training accuracy climb to around 99% and test accuracy to about 98–99%. The tiny gap is normal — a small amount of overfitting.

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

SNAPSHOT_STEPS = {0, 50, 100, 200, 500, 1000, 2000}

history = {'step': [], 'train_loss': [], 'test_accuracy': []}
filter_snapshots = []

def snapshot_conv1():
    """Return a copy of the first-layer filters, shape [8, 1, 3, 3]."""
    return model.conv1.weight.detach().cpu().numpy().copy()

def evaluate():
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    model.train()
    return correct / total

# Snapshot at step 0 (pure random init)
filter_snapshots.append({'step': 0, 'filters': snapshot_conv1().tolist()})

model.train()
step = 0
start = time.time()
for epoch in range(5):
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        step += 1

        if step in SNAPSHOT_STEPS:
            acc = evaluate()
            history['step'].append(step)
            history['train_loss'].append(loss.item())
            history['test_accuracy'].append(acc)
            filter_snapshots.append({'step': step, 'filters': snapshot_conv1().tolist()})
            print(f'  step {step:5d}  |  loss {loss.item():.4f}  |  test acc {acc:.4f}')

# Final snapshot
final_acc = evaluate()
history['step'].append(step)
history['train_loss'].append(loss.item())
history['test_accuracy'].append(final_acc)
filter_snapshots.append({'step': step, 'filters': snapshot_conv1().tolist()})

elapsed = time.time() - start
print(f'\nTraining finished in {elapsed:.1f}s.  Final test accuracy: {final_acc:.4f}')

## Step 6: Inspect — What Did the Model Learn?

Now the interesting part. The filters that started as random noise at step 0 should have evolved into recognizable patterns — edges, curves, fragments of digits.

We plot the 8 first-layer filters before and after training. Each filter is a 3×3 grid of numbers. Positive numbers light up pixels; negative numbers darken them. Training taught each filter to respond to a different kind of local pattern.

Nobody told the model to look for edges. It invented edge detection on its own — because edge detection is the fastest way to reduce classification loss on images. That's the central lesson of Episode 2: **architecture encodes assumptions about data, and the features emerge from training.**

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))

init_filters = np.array(filter_snapshots[0]['filters'])   # [8, 1, 3, 3]
final_filters = np.array(filter_snapshots[-1]['filters']) # [8, 1, 3, 3]

for i in range(8):
    axes[0, i].imshow(init_filters[i, 0], cmap='RdBu_r')
    axes[0, i].set_title(f'F{i} init', fontsize=9)
    axes[0, i].axis('off')
    axes[1, i].imshow(final_filters[i, 0], cmap='RdBu_r')
    axes[1, i].set_title(f'F{i} trained', fontsize=9)
    axes[1, i].axis('off')

plt.suptitle('First-layer conv filters: before (top) vs after (bottom) training', y=1.02)
plt.tight_layout()
plt.show()

## Step 7: Feature Maps on a Sample Digit

A filter alone doesn't mean much — you need to see what it *does* when it meets an image. A feature map is the output of a filter convolved across the image: bright spots mean the filter saw what it was looking for at that location.

Below we take a single test digit and show the 8 feature maps produced by the first conv layer. You'll see different filters activate on different parts of the digit — some on horizontal strokes, some on verticals, some on curves. The network will combine these in later layers to decide which digit it's looking at.

In [ ]:
sample_img, sample_label = test_data[7]
sample_x = sample_img.unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    conv1_out = F.relu(model.conv1(sample_x))  # [1, 8, 28, 28]

conv1_out = conv1_out.squeeze(0).cpu().numpy()

fig, axes = plt.subplots(1, 9, figsize=(14, 2))
axes[0].imshow(sample_img.squeeze(), cmap='gray')
axes[0].set_title(f'Input (label {sample_label})', fontsize=9)
axes[0].axis('off')
for i in range(8):
    axes[i + 1].imshow(conv1_out[i], cmap='viridis')
    axes[i + 1].set_title(f'F{i}', fontsize=9)
    axes[i + 1].axis('off')
plt.suptitle('First-layer feature maps for one test digit', y=1.05)
plt.tight_layout()
plt.show()

## Step 8: Export for the Browser App

The Neural Architecture Tour app needs two files:

1. **`precomputed.json`** — everything the app needs to render without running the model: the training loss curve, the filter snapshots across training, a few sample feature maps. The schema is documented in `artifacts/schema.md` next to this notebook.
2. **`model.onnx`** — the trained model in a portable format. The app loads this via `@huggingface/transformers` and runs predictions client-side when the user draws a digit.

ONNX export is one line in PyTorch: `torch.onnx.export(...)`. Under the hood it traces the model's forward pass with a dummy input and records every operation as a portable graph.

In [ ]:
# Build the precomputed.json artifact

# Pick a few test digits for sample predictions
sample_records = []
model.eval()
with torch.no_grad():
    for idx in [0, 1, 2, 5, 7, 11]:
        img, label = test_data[idx]
        x = img.unsqueeze(0).to(device)
        logits = model(x).cpu().numpy()[0]
        probs = np.exp(logits - logits.max())
        probs = probs / probs.sum()
        conv1_maps = F.relu(model.conv1(x)).squeeze(0).cpu().numpy()
        sample_records.append({
            'index': idx,
            'image': img.squeeze().cpu().numpy().tolist(),
            'true_label': int(label),
            'pred_label': int(logits.argmax()),
            'probabilities': probs.tolist(),
            'conv1_feature_maps': conv1_maps.tolist(),
        })

artifact = {
    'version': '1',
    'episode': '02-cnn',
    'model': {
        'name': 'SmallCNN',
        'input_shape': [1, 1, 28, 28],
        'num_classes': 10,
        'parameters': n_params,
        'architecture': '2xconv(8,16) + 2xfc(64,10)',
    },
    'training': {
        'dataset': 'MNIST',
        'optimizer': 'Adam',
        'learning_rate': 1e-3,
        'batch_size': 128,
        'epochs': 5,
        'final_test_accuracy': final_acc,
        'total_steps': step,
        'wall_clock_seconds': elapsed,
    },
    'history': history,
    'filter_snapshots': filter_snapshots,
    'samples': sample_records,
    'onnx_model_url': './model.onnx',
}

with open('precomputed.json', 'w') as f:
    json.dump(artifact, f)

size_kb = len(json.dumps(artifact)) / 1024
print(f'Wrote precomputed.json ({size_kb:.1f} KB)')
print(f'  - training history: {len(history["step"])} points')
print(f'  - filter snapshots: {len(filter_snapshots)} across training')
print(f'  - sample records:   {len(sample_records)}')

In [ ]:
# Export to ONNX and verify

model.eval()
dummy_input = torch.randn(1, 1, 28, 28, device=device)
torch.onnx.export(
    model,
    dummy_input,
    'model.onnx',
    input_names=['input'],
    output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)
print('Wrote model.onnx')

# Verify with onnxruntime
import onnx
import onnxruntime as ort

onnx_model = onnx.load('model.onnx')
onnx.checker.check_model(onnx_model)

sess = ort.InferenceSession('model.onnx', providers=['CPUExecutionProvider'])
test_x = sample_img.unsqueeze(0).cpu().numpy()
onnx_out = sess.run(None, {'input': test_x})[0]
with torch.no_grad():
    torch_out = model(sample_img.unsqueeze(0).to(device)).cpu().numpy()

max_diff = np.abs(onnx_out - torch_out).max()
print(f'ONNX vs PyTorch max output diff: {max_diff:.2e} (should be ~1e-6 or smaller)')
assert max_diff < 1e-4, 'ONNX export does not match PyTorch!'
print('✅ ONNX export verified.')

## Step 9: Get Your Artifacts Out of Colab

Two files were produced:

- `precomputed.json`
- `model.onnx`

To download them (if you're in Colab):

```python
from google.colab import files
files.download('precomputed.json')
files.download('model.onnx')
```

Then place them into the app at:

```
neural-architecture-tour/
  episodes/02-cnn/app/public/data/
    precomputed.json
    model.onnx
```

Commit, push, and the GitHub Pages deploy will pick them up. The app's interactive stops will then animate real training dynamics and run live inference on whatever you draw.

In [ ]:
# Uncomment to download from Colab:
# from google.colab import files
# files.download('precomputed.json')
# files.download('model.onnx')

## What's Next

- **See your model running in the browser:** [Episode 2 app](https://provandal.github.io/neural-architecture-tour/02-cnn/)
- **Previous episode (MLP):** [Binary Digit Trainer](https://provandal.github.io/binarydigittrainer/)
- **Series home:** [Neural Architecture Tour](https://provandal.github.io/neural-architecture-tour/)

### Things to try changing

- **Fewer or more filters in `conv1`.** What happens to accuracy? To the variety of learned filters?
- **Remove the pooling layers.** The model gets much bigger and slower. Does accuracy improve?
- **Add a third conv layer.** Does it help? MNIST is simple enough that extra depth is mostly wasted — but other datasets reward depth a lot.
- **Use `kernel_size=5` instead of `3`.** Bigger receptive fields per layer, but fewer filters for the same compute.